# 14: Can you actually pick a partner by decorrelation? (EXPLORATORY)

Within a dataset, less shared failure means more union benefit (partial correlation -0.94,
R2 0.98). Out of sample that link vanished. This notebook asks why, and whether the selection
rule works as a **decision** rather than as a correlation.

1. **How measurable is dependence?** The effective sample behind each estimate: shared missed
   PDR and severe eyes per pair. Reliability: validation dependence versus fresh dependence,
   split-half inside the fresh set, and a validation-sized fresh subsample for a fair comparison.
2. **Attenuation.** If reliability is r, an observed out-of-sample correlation is roughly the
   true one times sqrt(r). Given measured reliability, how much predictive power could survive.
3. **Policy simulation, the decision itself.** Fix an anchor model, rank its ten candidate
   partners on validation by each rule, choose one, then measure what that choice delivers on
   the fresh set:
   - least severity-weighted shared miss cost (the rule in question)
   - least dependence excess over independence
   - best partner on its own severe-case performance (marginal quality)
   - random partner
   - oracle: the partner that turns out best on fresh data (upper bound)

If the decorrelation rule beats random and approaches the oracle, it works. If it lands near
random while marginal quality does better, the practical advice changes.

25% burden, weights 500/200/15, matched seeds. CPU, about 15 minutes. Run cells 1-5.

In [1]:
# Cell 1: load and set up
import io, os, json, itertools, datetime, time
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
svc=gbuild('drive','v3')
F_PRED='1nCloJ5o9veZb2xJL_2EjdVHQXN31jy4w'; F_RES='1-HT4ONiwIfuBtC19s7QkDd8Euklh0Xon'; F_FIG='18eyHRWuqFcUrYiL8nPtpIalV7CreLkMK'
def dl(fid,p):
    with io.FileIO(p,'wb') as fh:
        d=MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid)); done=False
        while not done: _,done=d.next_chunk()
def find_id(n,f):
    r=svc.files().list(q=f"'{f}' in parents and name='{n}' and trashed=false", fields='files(id)').execute().get('files',[])
    return r[0]['id'] if r else None
def up(p,f,m):
    svc.files().create(body={'name':os.path.basename(p),'parents':[f]}, media_body=MediaFileUpload(p,mimetype=m), fields='id').execute()
    print('  uploaded', os.path.basename(p))

ARCHS=['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
       'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
       'retfound_binary','retfound_5class','retfound_adversarial']
SEEDS=[0,1,2]; W={4:500,3:200,2:15}; BURDEN=0.25
def tag(k): return '' if k==0 else f'_seed{k}'
def load(prefix, split):
    P, ref = {}, None
    for a in ARCHS:
        for k in SEEDS:
            fn=f'{prefix}{a}{tag(k)}_v2.csv'
            if not os.path.exists(fn): dl(find_id(fn,F_PRED), fn)
            d=pd.read_csv(fn); assert set(d['split'])=={split}
            if ref is None: ref=d[['image_id','patient_id','true_grade','true_label']].reset_index(drop=True)
            P[(a,k)]=d['predicted_probability'].to_numpy(np.float64)
    return P, ref
PV,RV=load('eyepacs_predictions_val_','val'); PF,RF=load('eyepacs_predictions_fresh_','fresh')
def ctx(R_):
    G=R_['true_grade'].to_numpy(int); L=R_['true_label'].to_numpy(int)
    return {'G':G,'L':L,'n':len(G),'R':int(round(BURDEN*len(G))),
            'w':np.where(L==1, np.where(G==4,W[4],np.where(G==3,W[3],W[2])), 0.0),
            'pid':R_['patient_id'].astype(str).to_numpy()}
CV=ctx(RV); CF=ctx(RF)
print(f"validation: {CV['n']} images, severe {(CV['G']>=3).sum()}, grade4 {(CV['G']==4).sum()}")
print(f"fresh:      {CF['n']} images, severe {(CF['G']>=3).sum()}, grade4 {(CF['G']==4).sum()}")

validation: 2900 images, severe 132, grade4 58
fresh:      15794 images, severe 712, grade4 328


In [2]:
# Cell 2: pair quantities on any index subset
def topR(p,R):
    m=np.zeros(len(p),bool); m[np.argsort(-p,kind='stable')[:R]]=True; return m

def pair_table(P, C, idx=None):
    G,L,w = (C['G'],C['L'],C['w']) if idx is None else (C['G'][idx],C['L'][idx],C['w'][idx])
    n=len(G); R=int(round(BURDEN*n)); g4=(G==4); sev=(G>=3)
    sub={k:(v if idx is None else v[idx]) for k,v in P.items()}
    REF={k:topR(v,R) for k,v in sub.items()}
    mc=lambda m: float(w[(~m)&(L==1)].sum())
    ng={g:int((G==g).sum()) for g in (2,3,4)}
    rows=[]
    for a,b in itertools.combinations(ARCHS,2):
        for s in SEEDS:
            ra,rb=REF[(a,s)],REF[(b,s)]
            shared=float(w[(~ra)&(~rb)&(L==1)].sum())
            si=0.0
            for g in (2,3,4):
                mg=(G==g)
                si += W[g]*ng[g]*(((~ra)&mg).sum()/max(1,ng[g]))*(((~rb)&mg).sum()/max(1,ng[g]))
            ca,cb=mc(ra),mc(rb); u=ra|rb; R_or=int(u.sum())
            base_p = sub[(a,s)] if ca<=cb else sub[(b,s)]
            rows.append({'a':a,'b':b,'seed':s,'shared_cost':shared,'dep_excess':shared-si,
                         'cost_a':ca,'cost_b':cb,'marg_min':min(ca,cb),
                         'adv_union_miss_only':min(ca,cb)-shared,
                         'adv_union_same_volume':mc(topR(base_p,R_or))-shared,
                         'shared_pdr_misses':int(((~ra)&(~rb)&g4).sum()),
                         'shared_severe_misses':int(((~ra)&(~rb)&sev).sum()),
                         'pdr_union_avoided':float(min(((~ra)&g4).sum(),((~rb)&g4).sum())-((~ra)&(~rb)&g4).sum())})
    return pd.DataFrame(rows)

t0=time.time(); TV=pair_table(PV,CV); TF=pair_table(PF,CF)
print(f'{time.time()-t0:.0f}s')
SAMP={'validation_mean_shared_pdr_misses':round(float(TV['shared_pdr_misses'].mean()),2),
      'validation_mean_shared_severe_misses':round(float(TV['shared_severe_misses'].mean()),2),
      'fresh_mean_shared_pdr_misses':round(float(TF['shared_pdr_misses'].mean()),2),
      'fresh_mean_shared_severe_misses':round(float(TF['shared_severe_misses'].mean()),2)}
print(json.dumps(SAMP, indent=2))
print('\nThis is the effective sample behind each dependence estimate.')

1s
{
  "validation_mean_shared_pdr_misses": 1.84,
  "validation_mean_shared_severe_misses": 5.75,
  "fresh_mean_shared_pdr_misses": 6.66,
  "fresh_mean_shared_severe_misses": 21.39
}

This is the effective sample behind each dependence estimate.


In [3]:
# Cell 3: reliability of the dependence estimate, and attenuation
key=['a','b','seed']
mv=TV.set_index(key); mf=TF.set_index(key); common=[i for i in mv.index if i in mf.index]
rel_val_fresh=float(spearmanr([mv.loc[i,'dep_excess'] for i in common],
                              [mf.loc[i,'dep_excess'] for i in common]).correlation)
pats=pd.unique(CF['pid']); rng=np.random.default_rng(20260922)
perm=rng.permutation(len(pats)); half=set(pats[perm[:len(pats)//2]])
m1=np.array([p in half for p in CF['pid']]); idx1=np.where(m1)[0]; idx2=np.where(~m1)[0]
T1=pair_table(PF,CF,idx1).set_index(key); T2=pair_table(PF,CF,idx2).set_index(key)
rel_split=float(spearmanr([T1.loc[i,'dep_excess'] for i in common],
                          [T2.loc[i,'dep_excess'] for i in common]).correlation)
small=set(pats[perm[:CV['n']//2]])
idxs=np.where(np.array([p in small for p in CF['pid']]))[0]
T3=pair_table(PF,CF,idxs).set_index(key)
rel_small=float(spearmanr([T3.loc[i,'dep_excess'] for i in common],
                          [mf.loc[i,'dep_excess'] for i in common]).correlation)
REL={'validation_vs_fresh_dependence':round(rel_val_fresh,3),
     'fresh_split_half_dependence':round(rel_split,3),
     'validation_sized_subsample_vs_full_fresh':round(rel_small,3),
     'implied_max_out_of_sample_correlation_if_true_is_0.94':
         round(0.94*float(np.sqrt(max(0.0,rel_val_fresh))),3)}
print(json.dumps(REL, indent=2))
print('\nLow reliability means the ranking you would select on is mostly noise, even when the')
print('underlying relationship is strong.')

{
  "validation_vs_fresh_dependence": 0.407,
  "fresh_split_half_dependence": 0.579,
  "validation_sized_subsample_vs_full_fresh": 0.75,
  "implied_max_out_of_sample_correlation_if_true_is_0.94": 0.6
}

Low reliability means the ranking you would select on is mostly noise, even when the
underlying relationship is strong.


In [4]:
# Cell 4: policy simulation, the decision you would actually make
def partner_view(T, anchor, s):
    out={}
    for _,r in T[T.seed==s].iterrows():
        if r.a==anchor: out[r.b]=r
        elif r.b==anchor: out[r.a]=r
    return out
def partner_cost(r, anchor):
    return r['cost_b'] if r['a']==anchor else r['cost_a']

rules=['least_weighted_dependence','least_dependence_excess','best_partner_marginal','random','oracle_fresh']
rows=[]; rng2=np.random.default_rng(11)
for anchor in ARCHS:
    for s in SEEDS:
        V=partner_view(TV,anchor,s); F=partner_view(TF,anchor,s)
        cands=[c for c in V if c in F]
        if len(cands)<3: continue
        picks={'least_weighted_dependence': min(cands, key=lambda c: V[c]['shared_cost']),
               'least_dependence_excess':   min(cands, key=lambda c: V[c]['dep_excess']),
               'best_partner_marginal':     min(cands, key=lambda c: partner_cost(V[c], anchor)),
               'random':                    cands[int(rng2.integers(0,len(cands)))],
               'oracle_fresh':              max(cands, key=lambda c: F[c]['adv_union_same_volume'])}
        for rule,c in picks.items():
            rows.append({'anchor':anchor,'seed':s,'rule':rule,'partner':c,
                         'fresh_adv_union_same_volume':float(F[c]['adv_union_same_volume']),
                         'fresh_adv_union_miss_only':float(F[c]['adv_union_miss_only']),
                         'fresh_pdr_avoided':float(F[c]['pdr_union_avoided'])})
POL=pd.DataFrame(rows)
S=POL.groupby('rule')[['fresh_adv_union_same_volume','fresh_adv_union_miss_only','fresh_pdr_avoided']].mean().round(1).reindex(rules)
rand=S.loc['random','fresh_adv_union_same_volume']; orc=S.loc['oracle_fresh','fresh_adv_union_same_volume']
S['pct_of_oracle_gain_over_random']=((S['fresh_adv_union_same_volume']-rand)/(orc-rand)*100).round(1)
print(S.to_string())
print('\n0% means no better than choosing at random; 100% means as good as knowing the answer.')

                           fresh_adv_union_same_volume  fresh_adv_union_miss_only  fresh_pdr_avoided  pct_of_oracle_gain_over_random
rule                                                                                                                                
least_weighted_dependence                       1764.2                     8491.4                6.8                             2.4
least_dependence_excess                         1835.9                     8603.5                6.8                             4.9
best_partner_marginal                           2035.0                     8074.7                6.0                            11.8
random                                          1695.0                     8475.8                6.4                             0.0
oracle_fresh                                    4570.8                    11537.6                9.8                           100.0

0% means no better than choosing at random; 100% means as good as kn

In [5]:
# Cell 5: figure and save
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(12.5,4.6))
ax[0].scatter([mv.loc[i,'dep_excess'] for i in common],[mf.loc[i,'adv_union_same_volume'] for i in common],s=14,c='C1')
ax[0].set_xlabel('validation dependence excess (weighted)'); ax[0].set_ylabel('fresh union advantage')
ax[0].set_title(f"out of sample (reliability {REL['validation_vs_fresh_dependence']})")
order=[r for r in rules if r in S.index]
ax[1].bar(range(len(order)),[S.loc[r,'fresh_adv_union_same_volume'] for r in order],color=['C0','C0','C2','grey','C3'])
ax[1].set_xticks(range(len(order))); ax[1].set_xticklabels([r.replace('_','\n') for r in order],fontsize=8)
ax[1].set_ylabel('fresh union advantage'); ax[1].set_title('partner selection rules')
fig.suptitle('Can decorrelation be used to choose a partner? (exploratory)')
fig.tight_layout(); fig.savefig('partner_selection_v3.png', dpi=200)

OUT={'generated_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'status':'EXPLORATORY',
     'effective_sample':SAMP,'reliability':REL,'policy_simulation':S.reset_index().to_dict(orient='records')}
with open('partner_selection_v3.json','w') as f: json.dump(OUT,f,indent=2,default=str)
POL.to_csv('partner_selection_detail_v3.csv',index=False)
up('partner_selection_v3.json',F_RES,'application/json'); up('partner_selection_detail_v3.csv',F_RES,'text/csv')
up('partner_selection_v3.png',F_FIG,'image/png')
print(json.dumps(OUT['reliability'],indent=2))

  uploaded partner_selection_v3.json
  uploaded partner_selection_detail_v3.csv
  uploaded partner_selection_v3.png
{
  "validation_vs_fresh_dependence": 0.407,
  "fresh_split_half_dependence": 0.579,
  "validation_sized_subsample_vs_full_fresh": 0.75,
  "implied_max_out_of_sample_correlation_if_true_is_0.94": 0.6
}
